### Description of the Notebook - Governance Officer

In [68]:
from pymongo import MongoClient
from collections import Counter
import pandas as pd
import numpy as np
import json
import re
import os
import hashlib


### Loading the clean datasets

In [69]:
data_path = os.path.join('data', 'processed')
try:
    app_df = pd.read_csv(os.path.join(data_path, 'applications_clean.csv'))
    spend_df = pd.read_csv(os.path.join(data_path, 'spending_behavior.csv'))
    
    print("Success, file loaded correctly")
except FileNotFoundError:
    print(f"Error, file not found{os.path.abspath(data_path)}")

Success, file loaded correctly


In [70]:
# Display the first 10 rows of the applications dataset
print("Applications Clean Dataset:")
display(app_df.head(10))

# Display the first 10 rows of the spending behavior dataset
print("\nSpending Behavior Dataset:")
display(spend_df.head(10))

Applications Clean Dataset:


,app_id,processing_timestamp,applicant_info_full_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_date_of_birth,applicant_info_zip_code,financials_annual_income,financials_credit_history_months,financials_debt_to_income,financials_savings_balance,decision_loan_approved,decision_rejection_reason,loan_purpose,decision_interest_rate,decision_approved_amount,notes,data_quality_flag,missing_fields,flag_note
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,NaN,NaN
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,NaN,NaN
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,app_275,NaN,Maria Miller,maria.miller67@outlook.com,417-25-4912,172.25.58.70,Female,1982-02-14,10019.0,110000.0,33,0.05,49933,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,app_099,NaN,Nicholas King,nicholas.king46@outlook.com,613-23-2503,10.62.62.45,Male,1990-01-28,10022.0,55000.0,61,0.17,30159,True,NaN,NaN,5.6,27000.0,NaN,NaN,NaN,NaN
7,app_246,NaN,Susan Rivera,susan.rivera74@gmail.com,176-97-1864,192.168.158.59,Female,1991-10-11,90223.0,82000.0,31,0.29,21809,True,NaN,auto,2.8,38000.0,NaN,NaN,NaN,NaN
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044.0,69000.0,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,RESUBMISSION,NaN,NaN,NaN
9,app_348,NaN,Michael Mitchell,michael.mitchell42@hotmail.com,100-94-8400,172.28.12.121,Male,1989-10-10,10080.0,55000.0,5,0.41,13794,False,insufficient_credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Spending Behavior Dataset:


,app_id,category,amount
0,app_200,Shopping,480
1,app_200,Rent,790
2,app_200,Alcohol,247
3,app_037,Rent,608
4,app_037,Dining,96
5,app_037,Healthcare,243
6,app_215,Rent,109
7,app_024,Fitness,575
8,app_184,Entertainment,463
9,app_275,Entertainment,571


### Merging the datasets

In [71]:
spend_df_renamed = spend_df.rename(columns={
    'category': 'spending_behaviour',
    'amount': 'spending_behavior_amount'
})

In [72]:
# Grouping by 'app_id' to count how many spending behaviors are assigned to each person
# We use the renamed spending dataframe from your previous step
behavior_counts = spend_df_renamed.groupby('app_id')['spending_behaviour'].count()

# Identifying the maximum number of behaviors for a single individual
max_behaviors = behavior_counts.max()

print(f"The maximum number of spending behaviors for a single person is: {max_behaviors}")

# Displaying the distribution to understand how many applicants have multiple records
print("\nFrequency distribution of spending behavior counts:")
print(behavior_counts.value_counts().sort_index(ascending=False))

The maximum number of spending behaviors for a single person is: 4

Frequency distribution of spending behavior counts:
spending_behaviour
4      2
3     78
2    162
1    258
Name: count, dtype: int64


In [73]:
# 1. Create a sequence number (rank) for each behavior per applicant
# cumcount() starts at 0, so we add 1 to get behavior_1, behavior_2, etc.
spend_df_renamed['seq'] = spend_df_renamed.groupby('app_id').cumcount() + 1

# 2. Pivot the table to move sequences from rows to columns
# This creates a table where each app_id has exactly one row
pivoted_spend = spend_df_renamed.pivot(
    index='app_id', 
    columns='seq', 
    values='spending_behaviour'
)

# 3. Rename columns to match your requirements (spending_behaviour_1, etc.)
pivoted_spend.columns = [f'spending_behaviour_{i}' for i in pivoted_spend.columns]

# 4. Final Merge: join the applications with the flattened spending data
# Using a left join ensures we keep all applicants, even those without spending data
merged_df = pd.merge(app_df, pivoted_spend, on='app_id', how='left')

# 5. Verification: Check the shape and first few rows
print(f"Final merged dataset shape: {merged_df.shape}")
print(f"New columns added: {[col for col in merged_df.columns if 'spending_behaviour_' in col]}")

# Displaying results for applicants with multiple behaviors to verify the fix
merged_df.head()

Final merged dataset shape: (500, 26)
New columns added: ['spending_behaviour_1', 'spending_behaviour_2', 'spending_behaviour_3', 'spending_behaviour_4']


,app_id,processing_timestamp,applicant_info_full_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_date_of_birth,applicant_info_zip_code,financials_annual_income,financials_credit_history_months,financials_debt_to_income,financials_savings_balance,decision_loan_approved,decision_rejection_reason,loan_purpose,decision_interest_rate,decision_approved_amount,notes,data_quality_flag,missing_fields,flag_note,spending_behaviour_1,spending_behaviour_2,spending_behaviour_3,spending_behaviour_4
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Shopping,Rent,Alcohol,NaN
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rent,Dining,Healthcare,NaN
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,NaN,NaN,Rent,NaN,NaN,NaN
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,NaN,NaN,Fitness,NaN,NaN,NaN
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Entertainment,NaN,NaN,NaN


In [74]:
# Display the first 10 rows of the applications dataset
print("Merged Dataset:")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
print("Merged Dataset (Full View):")
display(merged_df.head(10))

Merged Dataset:
Merged Dataset (Full View):


,app_id,processing_timestamp,applicant_info_full_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_date_of_birth,applicant_info_zip_code,financials_annual_income,financials_credit_history_months,financials_debt_to_income,financials_savings_balance,decision_loan_approved,decision_rejection_reason,loan_purpose,decision_interest_rate,decision_approved_amount,notes,data_quality_flag,missing_fields,flag_note,spending_behaviour_1,spending_behaviour_2,spending_behaviour_3,spending_behaviour_4
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Shopping,Rent,Alcohol,NaN
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rent,Dining,Healthcare,NaN
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,NaN,NaN,Rent,NaN,NaN,NaN
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,NaN,NaN,Fitness,NaN,NaN,NaN
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Entertainment,NaN,NaN,NaN
5,app_275,NaN,Maria Miller,maria.miller67@outlook.com,417-25-4912,172.25.58.70,Female,1982-02-14,10019.0,110000.0,33,0.05,49933,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Entertainment,NaN,NaN,NaN
6,app_099,NaN,Nicholas King,nicholas.king46@outlook.com,613-23-2503,10.62.62.45,Male,1990-01-28,10022.0,55000.0,61,0.17,30159,True,NaN,NaN,5.6,27000.0,NaN,NaN,NaN,NaN,Dining,NaN,NaN,NaN
7,app_246,NaN,Susan Rivera,susan.rivera74@gmail.com,176-97-1864,192.168.158.59,Female,1991-10-11,90223.0,82000.0,31,0.29,21809,True,NaN,auto,2.8,38000.0,NaN,NaN,NaN,NaN,Healthcare,NaN,NaN,NaN
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044.0,69000.0,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,RESUBMISSION,NaN,NaN,NaN,Insurance,Dining,NaN,NaN
9,app_348,NaN,Michael Mitchell,michael.mitchell42@hotmail.com,100-94-8400,172.28.12.121,Male,1989-10-10,10080.0,55000.0,5,0.41,13794,False,insufficient_credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fitness,Travel,NaN,NaN


In [75]:
# 1. Selezioniamo solo le 4 nuove colonne di spending behaviour
spending_cols = ['spending_behaviour_1', 'spending_behaviour_2', 'spending_behaviour_3', 'spending_behaviour_4']

# 2. Usiamo .stack() per unire i valori di tutte le colonne in una sola serie
# .stack() ignora automaticamente i valori NaN, contando solo i comportamenti reali
category_summary = merged_df[spending_cols].stack().value_counts().reset_index()

# 3. Rinominiamo le colonne per chiarezza, come nel tuo codice originale
category_summary.columns = ['Spending Behaviour', 'Absolute Frequency']

# 4. Visualizziamo il riepilogo
print("Summary of Spending Categories (Total across all 4 slots):")
display(category_summary)

Summary of Spending Categories (Total across all 4 slots):


,Spending Behaviour,Absolute Frequency
0,Travel,80
1,Utilities,76
2,Entertainment,72
3,Fitness,70
4,Healthcare,68
5,Insurance,67
6,Dining,65
7,Groceries,65
8,Education,64
9,Transportation,61


# PII IDENTIFICATION 

The given dataset contains the following PIIs:

**Direct Identifiers:**
- **Name, Email, and Social Security Number (SSN)** are direct identifiers as they can be uniquely and obviously linked to an individual without combining any further data.

**Quasi-identifiers:** This category encompasses all attributes that can lead to the identification of data subjects when combined with other fields; these identifiers, by themselves, do not lead to the obvious and direct identification of a person.
- **Gender, Date of Birth, and ZIP Code** are typical quasi-identifiers.
- **Annual Income, Savings Balance, Approved Loan Amount, and Interest Rate** are financial attributes that can be combined to identify a specific individual.
- **IP Address:** IP addresses fall into the quasi-identifier category because, on their own, they cannot lead to the direct identification of a data subject.

**Spending behavior** is not identified as a Personally Identifiable Information (PII) identifier, as its level of granularity appears too broad to lead to the identification of a data subject. It must be noted, however, that if NovaCred possessed more granular and capillary spending data, this category would certainly be included in the list of PIIs. Additionally, from the list of unique spending behaviors, we can identify  sensitive categories (Alcohol, Gambling, and Adult Entertainment) inside of this variable that clearly violate Article 9 of the GDPR, which protects the processing of sensitive data. This violation will be addressed later in a specific section of the GDPR mapping.

# GDPR MAPPING

In the following sections, Novacred data governance will be evaluated under the GDPR principles stated in Article 5.

### Lawfullness

**Lawfulness:** processing of personal data carried out by a controller must have a legal basis under the GDPR. In this particular business case, the **legal basis is the execution of a contract:** Novacred can legitimately store personal data for the potential execution of a loan contract requested by the client itself. Therefore, under this partuclar circumstance no violations can be found.

### Fairness

# MISSING PART

**Fairness**: the fairness principle states that the processing of personal data must be fair towards the individual whose personal data are concerned. As we discovered in the bias analysis part several biases have been found in the model, therefore we can confidently say that this principle was not respected.

### Transparency

The transparency principle states that data controllers must provide individuals with clear information regarding the processing of their personal data before it is provided. We cannot assess whether this principle is respected or not, as we do not possess any information regarding the processing terms and conditions released by the company. If missing, the company must provide a clear and exhaustive description of the types of data collected, the purpose of their processing, and the storage period.

### Purpose Limitation and Data Minimisation
The purpose limitation principle states that personal data must be collected for specified, explicit, and legitimate purposes. Among the PII attributes outlined above, the IP address stands out as a variable that is not strictly necessary for the credit risk assessment and loan concession process: consequently, its inclusion violates the purpose limitation principle.

To ensure compliance, IP addresses have been eliminated from the dataset, and future data ingestion processes should be modified to no longer request IP addresses from clients.

In [76]:
# --- GOVERNANCE OFFICER TASK: DATA MINIMIZATION ---
# Removing the IP address as it violates GDPR Art. 5(1)(c)

if 'applicant_info_ip_address' in merged_df.columns:
    merged_df = merged_df.drop(columns=['applicant_info_ip_address'])
    print("SUCCESS: 'applicant_info_ip_address' has been removed.")
else:
    print("NOTE: Column not found or already removed.")

# Verify the current columns
print("\nCurrent columns in app_df:")
print(merged_df.columns.tolist())

SUCCESS: 'applicant_info_ip_address' has been removed.

Current columns in app_df:
['app_id', 'processing_timestamp', 'applicant_info_full_name', 'applicant_info_email', 'applicant_info_ssn', 'applicant_info_gender', 'applicant_info_date_of_birth', 'applicant_info_zip_code', 'financials_annual_income', 'financials_credit_history_months', 'financials_debt_to_income', 'financials_savings_balance', 'decision_loan_approved', 'decision_rejection_reason', 'loan_purpose', 'decision_interest_rate', 'decision_approved_amount', 'notes', 'data_quality_flag', 'missing_fields', 'flag_note', 'spending_behaviour_1', 'spending_behaviour_2', 'spending_behaviour_3', 'spending_behaviour_4']


### Data minimisation

# EDIT THIS PART

### Accuracy

### Storage Limitation

The storage limitation principle dictates that personal data should be kept in a form which permits identification of data subjects for no longer than is necessary for the purposes for which the personal data are processed. Furthermore, explicit time limits must be established for the automatic erasure of customer records.

Currently, NovaCred lacks any defined time limits; all data is stored indefinitely, which represents a clear violation of this principle. To ensure compliance, the company must define a reasonable retention threshold (e.g., 1 year), after which data is subject to automatic deletion. In this regard, a distinction must be made based on the outcome of the application:

Approved Loans: The retention countdown shall only commence once the customer's legal and financial obligations toward the company are fully extinguished and recorded in the system. For instance, data should be deleted one year after the final loan installment has been successfully processed.

Denied Loans: For applicants whose loans were rejected, the one-year countdown shall start from the date the rejection notification is issued. In the event of a new submission, the countdown is reset until the issuance of a new decision.

In [77]:
# Assuming 'application_date' exists; if not, we use a placeholder for demonstration
if 'application_date' not in merged_df.columns:
    merged_df['application_date'] = pd.to_datetime('2024-01-01')
else:
    merged_df['application_date'] = pd.to_datetime(merged_df['application_date'])

# 1. Create 'loan_rejection_date'
# This field is populated only if the loan was denied
merged_df['loan_rejection_date'] = pd.NaT
merged_df.loc[merged_df['decision_loan_approved'] == False, 'loan_rejection_date'] = merged_df['application_date']

# 2. Create 'final_installment_date'
# For approved loans, we simulate the end of the contract (e.g., 2 years after application)
merged_df['final_installment_date'] = pd.NaT
merged_df.loc[merged_df['decision_loan_approved'] == True, 'final_installment_date'] = \
    merged_df['application_date'] + pd.DateOffset(years=2)

# 3. Create 'automatic_deletion_date'
# Adding a 365-day retention threshold (1 year) to the available trigger date
merged_df['automatic_deletion_date'] = \
    merged_df[['loan_rejection_date', 'final_installment_date']].max(axis=1) + pd.Timedelta(days=365)

# Verification: display the new retention management columns
print("Data Retention Audit Trail:")
display(merged_df[['decision_loan_approved', 'loan_rejection_date', 'final_installment_date', 'automatic_deletion_date']].head(10))

Data Retention Audit Trail:


,decision_loan_approved,loan_rejection_date,final_installment_date,automatic_deletion_date
0,False,2024-01-01,NaT,2024-12-31
1,False,2024-01-01,NaT,2024-12-31
2,True,NaT,2026-01-01,2027-01-01
3,True,NaT,2026-01-01,2027-01-01
4,False,2024-01-01,NaT,2024-12-31
5,False,2024-01-01,NaT,2024-12-31
6,True,NaT,2026-01-01,2027-01-01
7,True,NaT,2026-01-01,2027-01-01
8,False,2024-01-01,NaT,2024-12-31
9,False,2024-01-01,NaT,2024-12-31


### Integrity and confidentiality

**Violation of the principle of integrity:** the personal data managed by NovaCred are accessible in their original form, without any type of appropriate technical measure being implemented to secure the data. This deficiency represents a clear violation of Articles 25 and 32 of the GDPR, which oblige controllers to implement a range of protective measures to assure data safety.

To reinsure complaince with the european normatives, the following protective measures will be applied:
- **Pseudonymization of Name, Email, and SSN using Hashing with Salt:** We preferred pseudonymization over anonymization, as anonymization would have led to an irreversible loss of direct identifiers which are necessary information for a credit company. Among the various options, the hashing technique was chosen as it does not require the creation of a lookup table, the potential breach of which would lead to the exposure of all records; furthermore, the addition of salt—namely a random alphanumeric string at the beginning of each record—ensures that brute-force reversal cannot be used to trace back to the original data.

In [78]:
# 1. Define a secret salt (keep this separate and secure in a real scenario)
# A random alphanumeric string 
SECRET_SALT = "x7Km9pQr" 

def pseudonymize_value(value, salt=SECRET_SALT):
    """
    Hashes a string value using SHA-256 with an added salt.
    Ensures brute-force reversal is infeasible.
    """
    if pd.isna(value):
        return None
    
    # Step 1 & 3: Add salt + input, encode to bytes, and hash
    combined_string = salt + str(value).strip().lower()
    hashed_obj = hashlib.sha256(combined_string.encode())
    
    # Step 1: Return the hexadecimal representation
    return hashed_obj.hexdigest()

# 2. Applying the pseudonymization to the PII columns
# This protects Name, Email, and SSN while maintaining record linkability
pii_columns = ['applicant_info_full_name', 'applicant_info_email', 'applicant_info_ssn']

for col in pii_columns:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].apply(pseudonymize_value)
        print(f" Column '{col}' has been pseudonymized.")

# 3. Quick verification (displaying first rows of pseudonymized data)
merged_df[['applicant_info_full_name', 'applicant_info_email', 'applicant_info_ssn']].head(10)

 Column 'applicant_info_full_name' has been pseudonymized.
 Column 'applicant_info_email' has been pseudonymized.
 Column 'applicant_info_ssn' has been pseudonymized.


,applicant_info_full_name,applicant_info_email,applicant_info_ssn
0,1eba3321a1b29decf9f1d1327281254b395bfa9790e7b3d3c74715eed8e0b421,aa3e580fcd04b4d92f50effebef7236dc62123cde4a0129309202776bbd5a6a0,0b9028ba19acfed42365ea8e0654bf5794d05a3ac885060ce6ab6c50ad1b2b33
1,dd6b4961f706971b1e2027c9493a5c01aedf34525e964141b9a8376bcfbfc134,265658e9f71ef3473e67eabf1cc7c38eb337d061106094bd523056fb1f9ef6b8,ab5de87b02695c337b5d94721e3f705821c5adc2df80a5e3108fcd9786f6d1f4
2,7f5418e060c6ed1bfab9d639f9af47b70da813937ad76344be9f50b0ba892e9f,71abfd7702b1fd0436495b8ca22076d500c614d582e8152acd6b9a89efd993c5,6e0ca7ca8330acc6c76c34b704bcc1b3347b8e1313e6b3883030cb8f4718aa2d
3,b17a8ca15433d82186c50c0a0284c7a0558e02bcac92e3ea348a4b8cbfe63c45,a866c71ac09dbe2c3cc624aa0b69b60a29106e060defa7b1128db3870f454a6b,cc279cef70a644d891e3fe0678c30d64e01e63975494ff5059719af1aa12c996
4,5a724f38971e229df1264544fdcb615a4715b07f8eb8a93a8329cfbdea7031b5,614d8e92650ea199bcad35e1f9d621be1901f70b4f0595040d4fa6b237bb2101,c7598431a74193ff44e02912f3ddaa4a4a6966b201354f3d9a44879b83efb5f4
5,417e858154ba850523027cc97c5e20e8b3f9c523b7ac3c013e6e637724e74701,b3633f0579a49eec053cf67464386456965468e94090183ed052aec0651b786c,0516c5f7b35fc65933a2a08830fdf32f723e243c6d0998df1389bd9313a7b855
6,b9c8b40d084532ac1a1d503f5340c4f69368fb5ce54b929d35795e9029f91c74,4c69ad4a801b4593fe7b93de8235bbdab1b3f72332e46a97342a71286ed9f558,4a40acd889f953b7503225401faf46ebeac7297690950edf39398de1add5eada
7,51e8cb52d16389bf6718296fa994b2618b9e5f09d102ba6f3e8d7270510f9439,37c39a5b25be1550145e88f1feadaad67cef518806239c90d2def4548f7f5771,d4088dc1dfd780de273cc01e020097ac961c124ce0b678fe631942e46b2e43f3
8,52085061ba40aa79bf65b595b44ddf37b7d9693f085ae5f845116a3f65784791,409a3b94df1bdbe840ec85ead1648f1e4699f881dbdb5e994815f165736c423a,d2dc343181e247a85ed67a0b08286d5d47be3f496aa3efddb7dc171c143fa796
9,9cf8078b22e6fb583d4f3a953a396dd8fbae1854797c3f6fe27e20ae625cbc8c,80d429fade5ee7ee63eab6e361609f89c72576e81973f130fe8688d2fcbd4e6b,73cd77e87f3ab7a2c5ed97ed445c2ab49b7dee2b614fe7a452e2518b41dd06aa


### Article 9: processing of sensitive data

As anticipated in the PII Identification section, the variable Spending Behavior attribute contains several categories that must be considered with particular attention, as they fall, directly or indirectly, into the definition of senstive information according to article 9 of GDPR:

- **Healthcare:** This category is directly identifiable as carrying sensitive information, as it reveals an individual's health status. Spending patterns in this category can be clearly linked to a data subject having specific health conditions.
- **Alcohol:** This spending behavior can be indirectly linked to a person's health status; it is widely recognized that alcohol consumption is associated with specific health risks and conditions.
- **Adult Entertainment:** This category constitutes sensitive data as it reveals information regarding a person's sex life.
- **Gambling:** Gambling is associated with addictions that have repercussions on a person's physical and mental state. Therefore, it can be affirmed that gambling behavior may reveal clear information about an individual's health status, which falls under the categories of data explicitly defined as sensitive by the GDPR.

NovaCred's data processing does not fall into the legal exceptions under which sensitive information can be collected without explicit consent; therefore, it must be assessed whether explicit consent was provided by clients at the moment of submission. The provided data **completely lacks a consent tracking mechanism**, which represents a **clear violation of Article 9 of the GDPR.** To restore the lawfulness of the processed data, the following operations will be performed:

- **Deletion of historical sensitive data:** NovaCred is not subject to an absolute prohibition on using this sensitive data in loan granting decisions; this data can be used, provided that explicit consent is obtained. Therefore, the sensitive categories themselves will not be eliminated from the system; only the specific sensitive records collected in the past in violation of the law will be deleted.
- **Establishment of an explicit consent procedure:** Creation of a formal explicit consent request procedure: during the loan application submission phase, the client must be explicitly asked whether they grant their consent for the processing of sensitive data.
- **Implementation of a consent tracking attribute:** Creation of a new 'tracking_customer_consent' column: True if consent was given, False if not.

In [81]:
# Define sensitive categories to mitigate discriminatory bias in credit risk models
sensitive_categories = ['Healthcare', 'Alcohol', 'Adult Entertainment', 'Gambling']

# Dynamically identify all spending behavior columns generated during the flattening process
spending_cols = [col for col in merged_df.columns if 'spending_behaviour_' in col]

# Execute multi-column masking to redact sensitive information
for col in spending_cols:
    # Identify records matching sensitive categories within the specific column
    mask = merged_df[col].isin(sensitive_categories)
    
    # Redact sensitive values using NaN to ensure ethical compliance and data minimization
    merged_df.loc[mask, col] = np.nan

# Verification of the redacted feature set for the governance audit trail
merged_df[spending_cols].head()

,spending_behaviour_1,spending_behaviour_2,spending_behaviour_3,spending_behaviour_4
0,Shopping,Rent,NaN,NaN
1,Rent,Dining,NaN,NaN
2,Rent,NaN,NaN,NaN
3,Fitness,NaN,NaN,NaN
4,Entertainment,NaN,NaN,NaN


In [83]:
# Aggregate frequencies across all flattened spending columns
# .stack() is utilized to consolidate columns while automatically ignoring NaN values
category_summary = merged_df[spending_cols].stack().value_counts().reset_index()

# Formatting the summary dataframe for clear reporting
category_summary.columns = ['Spending Behaviour', 'Absolute Frequency']

# Displaying the final distribution for governance auditing
print("Consolidated Summary of Non-Sensitive Spending Categories:")
display(category_summary)

Consolidated Summary of Non-Sensitive Spending Categories:


,Spending Behaviour,Absolute Frequency
0,Travel,80
1,Utilities,76
2,Entertainment,72
3,Fitness,70
4,Insurance,67
5,Dining,65
6,Groceries,65
7,Education,64
8,Transportation,61
9,Rent,59


# Comment

# AI ACT MAPPING

### Risk Classification: High-Risk AI System


In accordance with article 5b of the EU AI Act, the NovaCred system is directly classified as a High-Risk AI System, as it falls into the category of AI technologies intended for evaluating the creditworthiness of natural persons or establishing their credit scores. Given that credit assessments significantly impact an individual's access to essential financial resources, the system is subject to stringent regulatory requirements to mitigate potential biases and ensure fundamental rights protection.

### Regulatory Requirements

Novacred must respect the following regulatory requirements under the AI Act:

- **Human Oversight (Article 14):** the credit scoring system must include a human-in-the-loop: indeed, human intervention must always be possible within the AI system, allowing a person to interrupt or override the decision made by the algorithm. This capability is particularly critical because the legal liability for the decisions made by the algorithm rests with a person, and obviously not with the algorithm itself.

The provided dataset contains no mention of a human-in-the-loop. To rectify this omission, we proceed to:

Rename the existing variable decision_loan_approved to the new variable decision_loan_approved_AI.

Create the variable decision_loan_approved_human, intended to track the presence (or absence) of human intervention in the decision-making process.

In [84]:
# 1. Renaming the algorithmic decision variable to clarify that the decision was taken by AI
if 'decision_loan_approved' in merged_df.columns:
    merged_df.rename(columns={'decision_loan_approved': 'decision_loan_approved_AI'}, inplace=True)

# 2. Initializing the human oversight tracking variable
# All values are set to NaN to represent the current absence of human-in-the-loop intervention
merged_df['decision_loan_approved_human'] = np.nan

# 3. Final verification of the restructured decision-making columns
# Displaying the first 10 observations to confirm the audit-ready structure
merged_df[['decision_loan_approved_AI', 'decision_loan_approved_human']].head(10)

,decision_loan_approved_AI,decision_loan_approved_human
0,False,NaN
1,False,NaN
2,True,NaN
3,True,NaN
4,False,NaN
5,False,NaN
6,True,NaN
7,True,NaN
8,False,NaN
9,False,NaN


- **Data and Data Governance (Article 10):** The AI system must be based on high-quality datasets that are subject to strict governance practices. This article specifically requires that training, validating, and testing data are relevant, representative, and to the best extent possible, free of errors. It is mandatory to ensure the system is not biased and does not lead to discrimination against data subjects.

**Record-keeping / Logging (Article 12):** High-risk AI systems must technically allow for the automatic recording of events (logs) throughout their lifetime. This ensures the traceability of the system's functioning and allows for effective monitoring of its outputs and potential risks.

**Accuracy, Robustness, and Cybersecurity (Article 15):** The AI system must be designed to achieve high levels of technical accuracy and resilience against errors, faults, or inconsistencies. It must also be robust against attempts by unauthorized third parties to alter its use or performance.